In [1]:
import sys
import polars as pl
import random, re
from scipy.stats import norm

In [2]:
# TODO also simulate different tails:
phosphorus = 30.973761998
oxygen = 31.989829239
d_ribose = 150.05282342
tail_5prime = [d_ribose, oxygen, 2 * oxygen + phosphorus, oxygen]

In [3]:
masses = pl.read_csv("../workflow/resources/masses_purines_modification.tsv",separator="\t")#, sep="\t").set_index("nucleoside", drop=False)
masses

nucleoside,monoisotopic_mass
str,f64
"""A""",267.09675
"""G""",283.09167
"""C""",243.08552
"""U""",244.06954
"""1A""",281.1124
…,…
"""10G""",409.1597
"""102G""",425.1547
"""105G""",538.2023


In [54]:
#masses2 = pl.read("../workflow/resources/Pyramidine_modifications.xls")#,separator="\t")
masses2 = pl.read_excel("../workflow/resources/Pyramidine_modifications.xlsx")

In [57]:
masses_pyrimidines = masses2.select(
    pl.col("Symbol").alias("nucleoside"),
    pl.col("Mass").cast(pl.Float64).alias("monoisotopic_mass"),
    )

In [4]:
masses3 = pl.read_csv("../workflow/resources/pyramidine_modifications_code_mod.tsv",separator="\t")

In [5]:
masses3

Standard_Symbol,Symbol,Nucleoside,Chemical Formula,Mass
str,str,str,str,f64
"""001C""","""Cf""","""2' fluorocytidine""","""C9 H12 N3 O4 F1""",245.081184
"""002U""","""Tmoe""","""5-methyl-2'-O-methoxyethylurid…","""C13 H20 N2 O7""",316.127051
"""003U""","""Uf""","""2' fluorouridine""","""C9 H11 N2 O5 F1""",246.0652
"""004C""","""m5Cmoe""","""5-methyl-2'-O-methoxyethylcyti…","""C13 H21 N3 O6""",315.143035
"""02U""","""s2Um""","""2-thio-2'-O-methyluridine""","""C10 H14 N2 O5 S1""",274.062343
…,…,…,…,…
"""G""","""G""","""guanosine""","""C10 H13 N5 O5""",283.091668
"""U""","""U""","""uridine""","""C9 H12 N2 O6""",244.069536
"""0503U""","""mcmo5Um""","""2'-O-methyluridine 5-oxyacetic…","""C13H18N2O9""",346.10123


In [6]:
masses_pyrimidines = masses3.select(
    pl.col("Standard_Symbol").alias("nucleoside"),
    pl.col("Mass").cast(pl.Float64).alias("monoisotopic_mass"),
    )

In [8]:
masses_pyrimidines

nucleoside,monoisotopic_mass
str,f64
"""001C""",245.081184
"""002U""",316.127051
"""003U""",246.0652
"""004C""",315.143035
"""02U""",274.062343
…,…
"""G""",283.091668
"""U""",244.069536
"""0503U""",346.10123


In [16]:
masses_pyrimidines.filter(pl.col("nucleoside").is_in(["T","A","C","G","U"]))

nucleoside,monoisotopic_mass
str,f64
"""A""",267.096754
"""C""",243.08552
"""G""",283.091668
"""U""",244.069536


In [17]:
masses.filter(pl.col("nucleoside").is_in(["T","A","C","G","U"]))

nucleoside,monoisotopic_mass
str,f64
"""A""",267.09675
"""G""",283.09167
"""C""",243.08552
"""U""",244.06954


In [ ]:
masses_pyrimidines.write_csv("../workflow/resources/masses_pyrimidines_modifications.tsv", separator="\t")

In [10]:
masses_all = masses.vstack(masses_pyrimidines)
masses_all

nucleoside,monoisotopic_mass
str,f64
"""A""",267.09675
"""G""",283.09167
"""C""",243.08552
"""U""",244.06954
"""1A""",281.1124
…,…
"""G""",283.091668
"""U""",244.069536
"""0503U""",346.10123


In [11]:
masses_all = masses_all.unique(pl.col("nucleoside"),keep='first',maintain_order=True)
masses_all

nucleoside,monoisotopic_mass
str,f64
"""A""",267.09675
"""G""",283.09167
"""C""",243.08552
"""U""",244.06954
"""1A""",281.1124
…,…
"""8U""",246.085186
"""9U""",244.069536
"""0503U""",346.10123


In [12]:
masses_all

nucleoside,monoisotopic_mass
str,f64
"""A""",267.09675
"""G""",283.09167
"""C""",243.08552
"""U""",244.06954
"""1A""",281.1124
…,…
"""8U""",246.085186
"""9U""",244.069536
"""0503U""",346.10123


In [25]:
masses_all.filter(pl.col("nucleoside").is_in(["T","A","C","G","U"]))

nucleoside,monoisotopic_mass
str,f64
"""C""",243.08552
"""A""",267.09675
"""G""",283.09167
"""U""",244.06954


In [13]:
masses_all.write_csv("../workflow/resources/masses_all.tsv", separator="\t")

In [123]:
nucleoside_re = re.compile(r"\d*[ACGU]")
true_sequence = nucleoside_re.findall("CG100GUU")
n_fragments = 10
print(true_sequence)

['C', 'G', '100G', 'U', 'U']


In [124]:
true_sequence = nucleoside_re.findall("CUA104GU1A")
print(true_sequence)
print(len(true_sequence))

['C', 'U', 'A', '104G', 'U', '1A']
6


In [72]:
# helper functions
def random_fragment():
    l = random.randint(0, len(true_sequence) - 1)
    r = random.randint(l + 1, len(true_sequence))
    return l, r

In [73]:
fragments = pl.from_records(
    [random_fragment() for _ in range(n_fragments)],
    schema=["left", "right"],
    orient="row",
)
fragments

left,right
i64,i64
5,6
3,4
4,6
1,2
2,6
2,5
2,5
4,6
1,5


In [74]:
true_fragment_masses = [
    sum(
        masses.filter(pl.col("nucleoside") == b)
        .select(pl.col("monoisotopic_mass"))
        .item()
        for b in true_sequence[fragment["left"] : fragment["right"]]
    )
    for fragment in fragments.iter_rows(named=True)
]
true_fragment_masses


[281.1124,
 571.2126,
 525.1819399999999,
 244.06954,
 1363.49129,
 1082.37889,
 1082.37889,
 525.1819399999999,
 1326.44843,
 244.06954]

In [75]:
len(true_sequence)

6

In [76]:
backbone_middle_addition = 1.
backbone_start_addition  = 2.
backbone_end_addition    = 0.

In [77]:
#Copying the fragment masses to a new list to add the backbone masses to the fragments.
true_fragment_masses_with_backbone = [elem for elem in true_fragment_masses]

# Add backbone masses to the fragments, based on the position of the fragment in the sequence!
for iter,fragment in enumerate(fragments.iter_rows(named=True)):
    
    if fragment["left"] == 0:
        true_fragment_masses_with_backbone[iter] += backbone_start_addition
    elif fragment["left"] != len(true_sequence)-1:
        true_fragment_masses_with_backbone[iter] += backbone_middle_addition
    
    if fragment["right"] == len(true_sequence)-1:
        true_fragment_masses_with_backbone[iter] += backbone_end_addition
    elif fragment["right"] != 0:
        true_fragment_masses_with_backbone[iter] += backbone_middle_addition
    
    true_fragment_masses_with_backbone[iter] += (
        backbone_middle_addition*max(fragment["right"]-fragment["left"]+1-2,0))

true_fragment_masses_with_backbone

[282.1124,
 573.2126,
 528.1819399999999,
 246.06954,
 1368.49129,
 1085.37889,
 1085.37889,
 528.1819399999999,
 1330.44843,
 246.06954]

In [78]:
#Copying the fragment masses to a new list to add the backbone masses to the fragments.
true_fragment_masses_with_backbone = [elem for elem in true_fragment_masses]
for iter,fragment in enumerate(fragments.iter_rows(named=True)):
    
        if fragment["left"] == 0:
            true_fragment_masses_with_backbone[iter] += backbone_start_addition #+ backbone_masses["Tag5prime"] #Can add a 5'tag mass if applicable here!
        else:
            true_fragment_masses_with_backbone[iter] += backbone_end_addition
        
        if fragment["right"] == len(true_sequence)-1:
            true_fragment_masses_with_backbone[iter] += 0. #+ backbone_masses["Tag3prime"] #Can add a 3'tag mass if applicable here!   
        
        true_fragment_masses_with_backbone[iter] += (
            backbone_middle_addition*max(fragment["right"]-fragment["left"],0))

true_fragment_masses_with_backbone

[282.1124,
 572.2126,
 527.1819399999999,
 245.06954,
 1367.49129,
 1085.37889,
 1085.37889,
 527.1819399999999,
 1330.44843,
 245.06954]

In [79]:
fragment_noise = norm.rvs(scale=0.0001, size=len(true_fragment_masses_with_backbone))
observed_fragment_masses = [
    max(mass + mass*noise, 0.0)
    for noise, mass in zip(fragment_noise, true_fragment_masses_with_backbone)
]

In [80]:
observed_fragment_masses

[np.float64(282.08665589825904),
 np.float64(572.2422980095666),
 np.float64(527.2003768387639),
 np.float64(245.0629225083547),
 np.float64(1367.5442006422852),
 np.float64(1085.242608301419),
 np.float64(1085.4643523355196),
 np.float64(527.266394509415),
 np.float64(1330.4226630976725),
 np.float64(245.06278371135858)]

In [81]:
fragments = fragments.with_columns(
    (pl.col("left") == 0).alias("is_start"),
    (pl.col("right") == (len(true_sequence))-1).alias("is_end"),
    (pl.col("right") == pl.col("left")).alias("single_nucleotide"),
    pl.Series(true_fragment_masses).alias("true_nucleoside_mass"),
    pl.Series(true_fragment_masses_with_backbone).alias("true_mass_with_backbone"),
    pl.Series(observed_fragment_masses).alias("observed_mass"),
)
fragments

left,right,is_start,is_end,single_nucleotide,true_nucleoside_mass,true_mass_with_backbone,observed_mass
i64,i64,bool,bool,bool,f64,f64,f64
5,6,false,false,false,281.1124,282.1124,282.086656
3,4,false,false,false,571.2126,572.2126,572.242298
4,6,false,false,false,525.18194,527.18194,527.200377
1,2,false,false,false,244.06954,245.06954,245.062923
2,6,false,false,false,1363.49129,1367.49129,1367.544201
2,5,false,true,false,1082.37889,1085.37889,1085.242608
2,5,false,true,false,1082.37889,1085.37889,1085.464352
4,6,false,false,false,525.18194,527.18194,527.266395
1,5,false,true,false,1326.44843,1330.44843,1330.422663


In [83]:
num_parts = 2
breakpoints = random.sample(range(1, len(true_sequence)), num_parts - 1)
breakpoints

[2]

In [84]:
true_sequence

['C', 'U', 'A', '104G', 'U', '1A']

In [85]:
def random_fragment(num_parts = num_parts):
    if num_parts == 1:
        breakagepoints = [int(0)]
    else:
        breakagepoints = sorted(random.sample(range(1, len(true_sequence)), num_parts - 1))
        #Implement the condition that the num_parts cannot be greater than the sequence length!
    return breakagepoints

In [86]:
[random_fragment(num_parts = random.randint(1,3)) for _ in range(n_fragments)]

[[2, 5], [0], [3, 4], [0], [3], [5], [0], [2], [0], [1, 3]]

In [87]:
[random.randint(2,2) for _ in range(n_fragments)]

[2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

In [88]:
breakagepoints = [random_fragment(num_parts = random.randint(1,3)) for _ in range(n_fragments)]
print(breakagepoints)

[[4], [0], [0], [1], [2, 4], [4, 5], [0], [3], [5], [4, 5]]


In [89]:
def generate_left_right(breakagepoints):
    left,right = [],[]
    for exp in breakagepoints:
        left.append(0)
        for part in exp:
            if part != 0:
                right.append(part-1)
                left.append(part)
        right.append(len(true_sequence)-1)
    return left,right
l,r = generate_left_right(breakagepoints)
print(l)
print(r)

[0, 4, 0, 0, 0, 1, 0, 2, 4, 0, 4, 5, 0, 0, 3, 0, 5, 0, 4, 5]
[3, 5, 5, 5, 0, 5, 1, 3, 5, 3, 4, 5, 5, 2, 5, 4, 5, 3, 4, 5]


In [90]:
l_breakage,r_breakage = generate_left_right(breakagepoints) 

# sample random fragments from true sequence
fragments = pl.from_records(
    list(zip(l_breakage,r_breakage)), 
    schema=["left", "right"],
    orient="row",
)
fragments

left,right
i64,i64
0,3
4,5
0,5
0,5
0,0
…,…
0,4
5,5
0,3


In [91]:
masses = pl.read_csv("../workflow/resources/masses_all.tsv", separator="\t")#.set_index("nucleoside", drop=False)

In [92]:
masses

nucleoside,monoisotopic_mass
str,f64
"""3483G""",508.1918
"""100G""",307.0917
"""68A""",297.1073
"""k2C""",371.180483
"""347G""",436.1706
…,…
"""2160A""",397.142
"""U""",244.06954
"""chm5U""",318.06993


In [93]:
true_fragment_masses = [
    sum(
        masses.filter(pl.col("nucleoside") == b)
        .select(pl.col("monoisotopic_mass"))
        .item()
        for b in true_sequence[fragment["left"] : fragment["right"] + 1]
    )
    for fragment in fragments.iter_rows(named=True)
]
true_fragment_masses

[1325.46441,
 525.1819399999999,
 1850.64635,
 1850.64635,
 243.08552,
 1607.56083,
 487.15506,
 838.30935,
 525.1819399999999,
 1325.46441,
 244.06954,
 281.1124,
 1850.64635,
 754.25181,
 1096.39454,
 1569.53395,
 281.1124,
 1325.46441,
 244.06954,
 281.1124]

In [94]:
fragment_noise = norm.rvs(scale=0.5, size=len(true_fragment_masses))
observed_fragment_masses = [
    max(true_mass + noise, 0.0)
    for noise, true_mass in zip(fragment_noise, true_fragment_masses)
]
observed_fragment_masses

[np.float64(1325.1654586469797),
 np.float64(524.9164994258801),
 np.float64(1851.5809250662728),
 np.float64(1850.9023043448587),
 np.float64(243.30128934158097),
 np.float64(1607.9861340849634),
 np.float64(487.27721846826444),
 np.float64(838.4015923597016),
 np.float64(525.0553179963348),
 np.float64(1325.001590357069),
 np.float64(243.8790698775186),
 np.float64(280.61205936000033),
 np.float64(1850.49191475471),
 np.float64(754.865532309867),
 np.float64(1095.4247926339694),
 np.float64(1569.7883426569829),
 np.float64(281.099669757661),
 np.float64(1326.1705253081047),
 np.float64(243.81499531326574),
 np.float64(281.3237343101226)]

In [99]:
fragments = fragments.with_columns(
    (pl.col("left") == 0).alias("is_start"),
    (pl.col("right") == (len(true_sequence))-1).alias("is_end"),
    (pl.col("right") == pl.col("left")).alias("single_nucleotide"),
    pl.Series(true_fragment_masses).alias("true_mass"),
    pl.Series(observed_fragment_masses).alias("observed_mass"),
)
fragments


left,right,is_start,is_end,single_nucleotide,true_mass,observed_mass
i64,i64,bool,bool,bool,f64,f64
0,3,true,false,false,1325.46441,1325.165459
4,5,false,true,false,525.18194,524.916499
0,5,true,true,false,1850.64635,1851.580925
0,5,true,true,false,1850.64635,1850.902304
0,0,true,false,true,243.08552,243.301289
…,…,…,…,…,…,…
0,4,true,false,false,1569.53395,1569.788343
5,5,false,true,true,281.1124,281.09967
0,3,true,false,false,1325.46441,1326.170525


In [101]:
fragments = fragments.with_columns(
    (pl.col("left") == 0).alias("is_start"),
    (pl.col("right") == (len(true_sequence))-1).alias("is_end"),
    (pl.col("right") == pl.col("left")).alias("single_nucleotide"),
    (true_sequence[pl.col("left")]:pl.col("right")).alias("sequence"),
    pl.Series(true_fragment_masses).alias("true_nucleoside_mass"),
    pl.Series(true_fragment_masses_with_backbone).alias("true_mass_with_backbone"),
    pl.Series(observed_fragment_masses).alias("observed_mass"),
)
fragments

SyntaxError: invalid syntax (229009960.py, line 5)

In [ ]:
true_sequence



['C', 'U', 'A', '104G', 'U', '1A']

In [ ]:
fragments.select(true_sequence[pl.select(pl.col("left")).item():pl.col("right")])

AttributeError: 'Expr' object has no attribute 'item'

In [125]:
fragment_sequences = [
    ''.join(true_sequence[fragment["left"] : fragment["right"] + 1])
    for fragment in fragments.iter_rows(named=True)
]

In [122]:
true_sequence

['C', 'U', 'A', '104G', 'U', '1A']

In [126]:
fragment_sequences

['CUA104G',
 'U1A',
 'CUA104GU1A',
 'CUA104GU1A',
 'C',
 'UA104GU1A',
 'CU',
 'A104G',
 'U1A',
 'CUA104G',
 'U',
 '1A',
 'CUA104GU1A',
 'CUA',
 '104GU1A',
 'CUA104GU',
 '1A',
 'CUA104G',
 'U',
 '1A']

In [118]:
fragments.with_columns(pl.col("left") + 1) #.map_elements(lambda s: true_sequence[s]))

#true_sequence[pl.select(pl.col("left")).item():pl.col("right")])

left,right,is_start,is_end,single_nucleotide,true_mass,observed_mass
i64,i64,bool,bool,bool,f64,f64
1,3,true,false,false,1325.46441,1325.165459
5,5,false,true,false,525.18194,524.916499
1,5,true,true,false,1850.64635,1851.580925
1,5,true,true,false,1850.64635,1850.902304
1,0,true,false,true,243.08552,243.301289
…,…,…,…,…,…,…
1,4,true,false,false,1569.53395,1569.788343
6,5,false,true,true,281.1124,281.09967
1,3,true,false,false,1325.46441,1326.170525


In [116]:
df = pl.DataFrame(
    {
        "a": [1, 2, 3],
        "b": [
            (2019, 1, 1),
            (2021, 1, 2),
            (2023, 1, 3),
        ],
        "c": [4.0, 5.0, 6.0],
        "d": ["a", "b", "c"],
    }
)


In [117]:
df.with_columns(
    (pl.col("a") * 2).alias("a_times_two"),
    pl.col("d").map_elements(lambda s: s + "x", return_dtype=str).alias("d_with_x"),
)

/var/folders/81/szh28xf51vn1q1xwmzrrlhy40000gn/T/ipykernel_20508/797264013.py:3: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("d").map_elements(lambda s: ...)
with this one instead:
  + pl.col("d") + 'x'

  pl.col("d").map_elements(lambda s: s + "x", return_dtype=str).alias("d_with_x"),


a,b,c,d,a_times_two,d_with_x
i64,list[i64],f64,str,i64,str
1,"[2019, 1, 1]",4.0,"""a""",2,"""ax"""
2,"[2021, 1, 2]",5.0,"""b""",4,"""bx"""
3,"[2023, 1, 3]",6.0,"""c""",6,"""cx"""


ValueError: can only call `.item()` if the dataframe is of shape (1, 1), or if explicit row/col values are provided; frame has shape (20, 1)

In [ ]:
seq
type(seq)

str

In [97]:
seq = []
for _ in range(10):
    seq.append(random.choice(["A", "U", "G", "C"])) 
seq
continuous_string = ''.join(seq)


'UAC'

In [110]:
masses.select(pl.col("nucleoside")).filter(pl.col("nucleoside").any not in ["U","A","G","C"])

TypeError: 'in <string>' requires string as left operand, not method

In [98]:
masses.select(pl.col("nucleoside"))

nucleoside
str
"""3483G"""
"""100G"""
"""68A"""
"""k2C"""
"""347G"""
…
"""2160A"""
"""U"""
"""chm5U"""


In [117]:
masses.filter((pl.col("nucleoside") not in ["U","A","G","C"]).all)

TypeError: the truth value of an Expr is ambiguous

You probably got here by using a Python standard library function instead of the native expressions API.
Here are some things you might want to try:
- instead of `pl.col('a') and pl.col('b')`, use `pl.col('a') & pl.col('b')`
- instead of `pl.col('a') in [y, z]`, use `pl.col('a').is_in([y, z])`
- instead of `max(pl.col('a'), pl.col('b'))`, use `pl.max_horizontal(pl.col('a'), pl.col('b'))`


In [14]:
masses.filter(~pl.col("nucleoside").is_in(["U","A","G","C"])).select(pl.col("nucleoside"))
modified_nucleoside_names = masses.filter(~pl.col("nucleoside").is_in(["U","A","G","C"])).select(pl.col("nucleoside")).to_series().to_list()
modified_nucleoside_names

['3483G',
 '100G',
 '68A',
 'k2C',
 '347G',
 'm5Um',
 '102G',
 'acp3D',
 '06A',
 'Tmoe',
 '019A',
 'm5U',
 'mcm5Um',
 '348G',
 '19A',
 '066A',
 'cmnm5se2U',
 'cmnm5ges2U',
 'm5s2U',
 'm1Y',
 'mcm5s2U',
 'm3Um',
 's2U',
 '2163A',
 'tm5U',
 'Cm',
 'cmnm5U',
 '34G',
 'mnm5s2U',
 'acp3Y',
 'm5D',
 '103G',
 'Ym',
 '2162A',
 'C+',
 '01G',
 '2165A',
 '1A',
 '1G',
 '0A',
 'inm5U',
 '69A',
 'ges2U',
 'ho5U',
 '342G',
 'nm5s2U',
 '3470G',
 'mnm5se2U',
 '62A',
 '00G',
 '3480G',
 'cm5s2U',
 'mcm5U',
 '0G',
 '2G',
 'D',
 '105G',
 '2164A',
 'tm5s2U',
 '22G',
 'f5Cm',
 'cnm5U',
 '66A',
 '65A',
 'm3Y',
 'm4Cm',
 's2Um',
 'acp3U',
 'm5Cmoe',
 '60A',
 '2A',
 '27G',
 'nm5se2U',
 '104G',
 '4G',
 'mnm5ges2U',
 'f5C',
 'hm5C',
 '2161A',
 'Uf',
 '106G',
 '61A',
 '227G',
 '34830G',
 'cmnm5Um',
 'm5Cm',
 'mchm5U',
 'm4,4Cm',
 '10G',
 'nchm5U',
 'nm5U',
 'Cf',
 'inm5Um',
 'ncm5s2U',
 '67A',
 '8A',
 'ac4C',
 '101G',
 'm4,4C',
 'Um',
 'm1acp3Y',
 '6A',
 'mcmo5U',
 '47G',
 '42G',
 '02G',
 'mchm5Um',
 's2C',
 'ac4C

In [13]:
len(nucleoside_list)

139

In [44]:
mutation_rate = 0.1
weights = [(1.-mutation_rate/len(modified_nucleoside_names))/4]*4 + [mutation_rate/len(modified_nucleoside_names)]*len(modified_nucleoside_names)


In [45]:
weights

[0.2498201438848921,
 0.2498201438848921,
 0.2498201438848921,
 0.2498201438848921,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0007194244604316547,
 0.0

In [26]:
sum(weights)

1.0009928057553956

In [ ]:
''.join([random.choices(["A", "U", "G", "C"] + modified_nucleoside_names , weights = weights ) for _ in range(10)])

TypeError: sequence item 0: expected str instance, list found

In [32]:
print(["A", "U", "G", "C"] + modified_nucleoside_names)

['A', 'U', 'G', 'C', '3483G', '100G', '68A', 'k2C', '347G', 'm5Um', '102G', 'acp3D', '06A', 'Tmoe', '019A', 'm5U', 'mcm5Um', '348G', '19A', '066A', 'cmnm5se2U', 'cmnm5ges2U', 'm5s2U', 'm1Y', 'mcm5s2U', 'm3Um', 's2U', '2163A', 'tm5U', 'Cm', 'cmnm5U', '34G', 'mnm5s2U', 'acp3Y', 'm5D', '103G', 'Ym', '2162A', 'C+', '01G', '2165A', '1A', '1G', '0A', 'inm5U', '69A', 'ges2U', 'ho5U', '342G', 'nm5s2U', '3470G', 'mnm5se2U', '62A', '00G', '3480G', 'cm5s2U', 'mcm5U', '0G', '2G', 'D', '105G', '2164A', 'tm5s2U', '22G', 'f5Cm', 'cnm5U', '66A', '65A', 'm3Y', 'm4Cm', 's2Um', 'acp3U', 'm5Cmoe', '60A', '2A', '27G', 'nm5se2U', '104G', '4G', 'mnm5ges2U', 'f5C', 'hm5C', '2161A', 'Uf', '106G', '61A', '227G', '34830G', 'cmnm5Um', 'm5Cm', 'mchm5U', 'm4,4Cm', '10G', 'nchm5U', 'nm5U', 'Cf', 'inm5Um', 'ncm5s2U', '67A', '8A', 'ac4C', '101G', 'm4,4C', 'Um', 'm1acp3Y', '6A', 'mcmo5U', '47G', '42G', '02G', 'mchm5Um', 's2C', 'ac4Cm', 'm4C', 'ncm5U', 'ho5C', '662A', '64A', 'm3U', 'cm5U', 'inm5s2U', 'mo5U', '01A', 'm3C

In [43]:
''.join(random.choices(["A", "U", "G", "C"] + modified_nucleoside_names , weights = weights, k=10))

'CUUCGGCGAU'

In [38]:
''.join([random.choices(["A", "U", "G", "C"] + modified_nucleoside_names , weights = weights ) for _ in range(10)])

TypeError: sequence item 0: expected str instance, list found

In [39]:
[random.choice(["A", "U", "G", "C"]) for _ in range(10)]

['U', 'A', 'U', 'G', 'G', 'G', 'A', 'G', 'U', 'A']

In [40]:
random.choice(["A", "U", "G", "C"])

'G'

In [46]:
ext = pl.read_csv("../results/simulation/CAGCAAAACGAUCAUAUAGAGAUCCGCAGU/30.tsv", separator="\t")

In [47]:
ext

left,right,is_start(5'),is_end(3'),single_nucleoside,true_nucleoside_mass,true_mass_with_backbone,observed_mass
i64,i64,bool,bool,bool,f64,f64,f64
0,11,true,false,false,3142.08994,3823.708086,3823.768696
12,29,false,true,false,4683.56742,5798.942568,5798.88407
0,11,true,false,false,3142.08994,3823.708086,3823.718764
12,29,false,true,false,4683.56742,5798.942568,5798.917598
0,29,true,true,false,7825.65736,9622.650654,9622.537301
…,…,…,…,…,…,…,…
0,29,true,true,false,7825.65736,9622.650654,9622.518846
0,29,true,true,false,7825.65736,9622.650654,9622.666081
0,29,true,true,false,7825.65736,9622.650654,9622.664509


In [48]:
sdf = []

In [49]:
sdf + ["asd"]

['asd']

In [70]:
true_sequence

['C', 'U', 'A', '104G', 'U', '1A']